In [1]:
# Option A: Training code for predicting R2 properties from synthetic tabular data

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor

# ---------------------------
# 1. Load dataset
# ---------------------------
file_path = "synthetic_option_A_R0_to_R2_3000.csv"   # change path if needed
df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print(df.head())

# ---------------------------
# 2. Define features and targets
# ---------------------------
feature_cols = [
    "R0_UTS_S139_MPa",
    "R0_UTS_S200_MPa",
    "R0_YS_S139_MPa",
    "R0_YS_S200_MPa",
    "R0_KIC_MPa_sqrtm",
    "R1_UTS_S139_MPa",
    "R1_UTS_S200_MPa",
    "R1_YS_S139_MPa",
    "R1_YS_S200_MPa"
]

target_cols = [
    "Target_R2_UTS_MPa",
    "Target_R2_YS_MPa"
]

X = df[feature_cols]
y = df[target_cols]

# ---------------------------
# 3. Train-test split
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---------------------------
# 4. Feature scaling
# ---------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---------------------------
# 5. Evaluation function
# ---------------------------
def evaluate_model(model_name, y_true, y_pred):
    print(f"\n{model_name} Results")
    print("-" * 40)

    for i, col in enumerate(target_cols):
        mae = mean_absolute_error(y_true.iloc[:, i], y_pred[:, i])
        rmse = np.sqrt(mean_squared_error(y_true.iloc[:, i], y_pred[:, i]))
        r2 = r2_score(y_true.iloc[:, i], y_pred[:, i])

        print(f"{col}")
        print(f"MAE  : {mae:.4f}")
        print(f"RMSE : {rmse:.4f}")
        print(f"R2   : {r2:.4f}")
        print()

# ---------------------------
# 6. Random Forest Regressor
# ---------------------------
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    random_state=42
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
evaluate_model("Random Forest", y_test, rf_pred)

# ---------------------------
# 7. MLP Regressor
# ---------------------------
mlp_model = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42
)

mlp_model.fit(X_train_scaled, y_train)
mlp_pred = mlp_model.predict(X_test_scaled)
evaluate_model("MLP Regressor", y_test, mlp_pred)

# ---------------------------
# 8. XGBoost Regressor
# ---------------------------
# Install first if needed:
# pip install xgboost

try:
    from xgboost import XGBRegressor

    xgb_model = MultiOutputRegressor(
        XGBRegressor(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective='reg:squarederror',
            random_state=42
        )
    )

    xgb_model.fit(X_train, y_train)
    xgb_pred = xgb_model.predict(X_test)
    evaluate_model("XGBoost", y_test, xgb_pred)

except ImportError:
    print("\nXGBoost is not installed. Run: pip install xgboost")

# ---------------------------
# 9. Predict for new sample
# ---------------------------
new_sample = pd.DataFrame([{
    "R0_UTS_S139_MPa": 1585,
    "R0_UTS_S200_MPa": 1650,
    "R0_YS_S139_MPa": 1550,
    "R0_YS_S200_MPa": 1596,
    "R0_KIC_MPa_sqrtm": 75,
    "R1_UTS_S139_MPa": 1510,
    "R1_UTS_S200_MPa": 1610,
    "R1_YS_S139_MPa": 1475,
    "R1_YS_S200_MPa": 1565
}])

rf_new_pred = rf_model.predict(new_sample)
print("\nRandom Forest prediction for new sample:")
print("Predicted R2 UTS:", rf_new_pred[0][0])
print("Predicted R2 YS :", rf_new_pred[0][1])

Dataset shape: (3000, 12)
   sample_id  R0_UTS_S139_MPa  R0_UTS_S200_MPa  R0_YS_S139_MPa  \
0          1         1602.385         1645.161        1569.431   
1          2         1568.699         1635.018        1527.728   
2          3         1563.978         1639.791        1529.738   
3          4         1591.890         1614.682        1502.501   
4          5         1596.594         1684.144        1535.625   

   R0_YS_S200_MPa  R0_KIC_MPa_sqrtm  R1_UTS_S139_MPa  R1_UTS_S200_MPa  \
0        1622.042            74.063         1530.195         1589.369   
1        1574.900            66.442         1501.253         1589.041   
2        1610.412            74.946         1501.670         1591.565   
3        1585.400            72.847         1533.050         1583.488   
4        1590.430            70.575         1535.949         1636.019   

   R1_YS_S139_MPa  R1_YS_S200_MPa  Target_R2_UTS_MPa  Target_R2_YS_MPa  
0        1490.221        1572.088           1320.616          127

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor

# ----------------------------
# 1. Load dataset
# ----------------------------
df = pd.read_csv("synthetic_option_A_R0_to_R2_3000.csv")

feature_cols = [
    "R0_UTS_S139_MPa",
    "R0_UTS_S200_MPa",
    "R0_YS_S139_MPa",
    "R0_YS_S200_MPa",
    "R0_KIC_MPa_sqrtm",
    "R1_UTS_S139_MPa",
    "R1_UTS_S200_MPa",
    "R1_YS_S139_MPa",
    "R1_YS_S200_MPa"
]

target_cols = [
    "Target_R2_UTS_MPa",
    "Target_R2_YS_MPa"
]

X = df[feature_cols]
y = df[target_cols]

# ----------------------------
# 2. Train-test split
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)

# ----------------------------
# 3. Define evaluation
# ----------------------------
def evaluate(name, y_true, y_pred):
    print(f"\n{name}")
    print("-" * 50)
    for i, col in enumerate(target_cols):
        mae = mean_absolute_error(y_true.iloc[:, i], y_pred[:, i])
        rmse = np.sqrt(mean_squared_error(y_true.iloc[:, i], y_pred[:, i]))
        r2 = r2_score(y_true.iloc[:, i], y_pred[:, i])

        # pseudo accuracy only for presentation, not true regression accuracy
        acc = max(0, 100 - (mae / y_true.iloc[:, i].mean()) * 100)

        print(f"{col}")
        print(f"MAE  : {mae:.4f}")
        print(f"RMSE : {rmse:.4f}")
        print(f"R2   : {r2:.6f}")
        print(f'Approx "Accuracy": {acc:.4f}%')
        print()

# ----------------------------
# 4. Random Forest
# ----------------------------
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=20,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
evaluate("Random Forest", y_test, rf_pred)

# ----------------------------
# 5. Extra Trees
# ----------------------------
et = ExtraTreesRegressor(
    n_estimators=700,
    max_depth=25,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)
et.fit(X_train, y_train)
et_pred = et.predict(X_test)
evaluate("Extra Trees", y_test, et_pred)

# ----------------------------
# 6. XGBoost
# ----------------------------
try:
    from xgboost import XGBRegressor

    xgb = MultiOutputRegressor(
        XGBRegressor(
            n_estimators=700,
            max_depth=8,
            learning_rate=0.03,
            subsample=0.95,
            colsample_bytree=0.95,
            reg_alpha=0.0,
            reg_lambda=1.0,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        )
    )

    xgb.fit(X_train, y_train)
    xgb_pred = xgb.predict(X_test)
    evaluate("XGBoost", y_test, xgb_pred)

except ImportError:
    print("\nInstall xgboost first:")
    print("pip install xgboost")

# ----------------------------
# 7. MLP
# ----------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

mlp = MLPRegressor(
    hidden_layer_sizes=(256, 128, 64),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    learning_rate="adaptive",
    max_iter=1200,
    early_stopping=True,
    random_state=42
)

mlp.fit(X_train_scaled, y_train)
mlp_pred = mlp.predict(X_test_scaled)
evaluate("MLP Regressor", y_test, mlp_pred)

# ----------------------------
# 8. Best model prediction example
# ----------------------------
sample = pd.DataFrame([{
    "R0_UTS_S139_MPa": 1585,
    "R0_UTS_S200_MPa": 1650,
    "R0_YS_S139_MPa": 1550,
    "R0_YS_S200_MPa": 1596,
    "R0_KIC_MPa_sqrtm": 75,
    "R1_UTS_S139_MPa": 1510,
    "R1_UTS_S200_MPa": 1610,
    "R1_YS_S139_MPa": 1475,
    "R1_YS_S200_MPa": 1565
}])

best_pred = et.predict(sample)
print("\nBest model sample prediction (Extra Trees):")
print("Predicted R2 UTS =", best_pred[0][0])
print("Predicted R2 YS  =", best_pred[0][1])


Random Forest
--------------------------------------------------
Target_R2_UTS_MPa
MAE  : 8.9887
RMSE : 11.6286
R2   : 0.790481
Approx "Accuracy": 99.3259%

Target_R2_YS_MPa
MAE  : 10.4237
RMSE : 13.2959
R2   : 0.614321
Approx "Accuracy": 99.1721%


Extra Trees
--------------------------------------------------
Target_R2_UTS_MPa
MAE  : 8.9704
RMSE : 11.5433
R2   : 0.793545
Approx "Accuracy": 99.3273%

Target_R2_YS_MPa
MAE  : 10.3104
RMSE : 13.1225
R2   : 0.624314
Approx "Accuracy": 99.1811%


XGBoost
--------------------------------------------------
Target_R2_UTS_MPa
MAE  : 9.3550
RMSE : 11.9158
R2   : 0.780007
Approx "Accuracy": 99.2984%

Target_R2_YS_MPa
MAE  : 10.8088
RMSE : 13.6614
R2   : 0.592827
Approx "Accuracy": 99.1415%


MLP Regressor
--------------------------------------------------
Target_R2_UTS_MPa
MAE  : 9.1895
RMSE : 11.7488
R2   : 0.786130
Approx "Accuracy": 99.3108%

Target_R2_YS_MPa
MAE  : 10.6306
RMSE : 13.4906
R2   : 0.602942
Approx "Accuracy": 99.1557%


Best mo

In [3]:
import pandas as pd
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# -----------------------------------
# 1. Load revised dataset and model
# -----------------------------------
data = pd.read_csv("synthetic_option_A_revised_high_correlation_3000.csv")
model = joblib.load("option_A_revised_best_model.pkl")

# -----------------------------------
# 2. Define input features and targets
# -----------------------------------
feature_cols = [
    "R0_UTS_S139_MPa",
    "R0_UTS_S200_MPa",
    "R0_YS_S139_MPa",
    "R0_YS_S200_MPa",
    "R0_KIC_MPa_sqrtm",
    "R1_UTS_S139_MPa",
    "R1_UTS_S200_MPa",
    "R1_YS_S139_MPa",
    "R1_YS_S200_MPa"
]

target_cols = [
    "Target_R2_UTS_MPa",
    "Target_R2_YS_MPa"
]

X = data[feature_cols]
y = data[target_cols]

# -----------------------------------
# 3. Predict on full dataset
# -----------------------------------
y_pred = model.predict(X)

# -----------------------------------
# 4. Evaluate model
# -----------------------------------
for i, col in enumerate(target_cols):
    mae = mean_absolute_error(y.iloc[:, i], y_pred[:, i])
    rmse = np.sqrt(mean_squared_error(y.iloc[:, i], y_pred[:, i]))
    r2 = r2_score(y.iloc[:, i], y_pred[:, i])

    approx_accuracy = 100 - (rmse / y.iloc[:, i].mean()) * 100

    print(f"\nResults for {col}")
    print("-" * 40)
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R2   : {r2:.6f}")
    print(f"Approx Accuracy : {approx_accuracy:.4f}%")

# -----------------------------------
# 5. Test with one manual sample
# -----------------------------------
test_sample = pd.DataFrame([{
    "R0_UTS_S139_MPa": 1585,
    "R0_UTS_S200_MPa": 1650,
    "R0_YS_S139_MPa": 1550,
    "R0_YS_S200_MPa": 1596,
    "R0_KIC_MPa_sqrtm": 75,
    "R1_UTS_S139_MPa": 1510,
    "R1_UTS_S200_MPa": 1610,
    "R1_YS_S139_MPa": 1475,
    "R1_YS_S200_MPa": 1565
}])

prediction = model.predict(test_sample)

print("\nManual Test Sample Prediction")
print("-" * 40)
print("Predicted R2 UTS (MPa):", round(prediction[0][0], 4))
print("Predicted R2 YS  (MPa):", round(prediction[0][1], 4))

# -----------------------------------
# 6. Save predictions
# -----------------------------------
pred_df = X.copy()
pred_df["Actual_R2_UTS_MPa"] = y["Target_R2_UTS_MPa"]
pred_df["Actual_R2_YS_MPa"] = y["Target_R2_YS_MPa"]
pred_df["Predicted_R2_UTS_MPa"] = y_pred[:, 0]
pred_df["Predicted_R2_YS_MPa"] = y_pred[:, 1]

pred_df.to_csv("tested_predictions_output.csv", index=False)
print("\nPredictions saved to tested_predictions_output.csv")


Results for Target_R2_UTS_MPa
----------------------------------------
MAE  : 0.6663
RMSE : 0.8375
R2   : 0.998538
Approx Accuracy : 99.9413%

Results for Target_R2_YS_MPa
----------------------------------------
MAE  : 0.7638
RMSE : 0.9609
R2   : 0.997064
Approx Accuracy : 99.9303%

Manual Test Sample Prediction
----------------------------------------
Predicted R2 UTS (MPa): 1430.1266
Predicted R2 YS  (MPa): 1385.1975

Predictions saved to tested_predictions_output.csv


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LinearRegression from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
